# Model design from ST-GCN model from the website - retraining - Spatial Temporal Graph Convolutional Networks (ST-GCN) for Skeleton-Based Action Recognition in PyTorch

https://github.com/yysijie/st-gcn

In [1]:
from Pose_Preprocessing_Pipeline_pasp import *

Removed 19 videos due to missing frames:
['semantic_segmentation_PA027_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA366_UGS_WoJ_2_DensePose_landmarks', 'semantic_segmentation_PA250_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA239_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA203_UGS_WoJ_1_DensePose_landmarks', 'semantic_segmentation_PA016_FGS_WoJ_2_DensePose_landmarks', 'semantic_segmentation_PA337_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA292_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA140_UGS_WoJ_2_DensePose_landmarks', 'semantic_segmentation_PA267_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA375_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA337_UGS_WoJ_2_DensePose_landmarks', 'semantic_segmentation_PA031_UGS_WoJ_2_DensePose_landmarks', 'semantic_segmentation_PA336_UGS_WJ_2_DensePose_landmarks', 'semantic_segmentation_PA026_FGS_WoJ_2_DensePose_landmarks', 'semantic_segmentation_PA383_UGS_WJ_2_DensePose_lan

In [2]:
# get the patient_name_clean
# Extract video_id from window_ids_clean (assuming format: "video_id_winXXX_fYYY-ZZZ")
video_ids_clean = [w.split("_win")[0] for w in window_ids_clean]

# Build mapping from df_video: video_id -> patient_name
video_to_patient = dict(zip(df_video["video_id"], df_video["patient_name"]))

# Map window_ids to patient names
patient_names_clean = np.array([video_to_patient[vid] for vid in video_ids_clean])
print(f"Patient names for QC-clean windows: {patient_names_clean.shape}")
print(f"Unique patients: {len(np.unique(patient_names_clean))}")


Patient names for QC-clean windows: (15001,)
Unique patients: 409


In [3]:
# %%
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# -------------------------------
# Assume after QC you have:
# X_clean: (N_windows, T, J, 3)
# y_binary_clean: (N_windows,)
# y_multilabel_clean: (N_windows, 5)
# window_ids_clean: list of window IDs
# patient_names_clean: list/array of patient_name per window
# -------------------------------

# Example:
# patient_names_clean = df_window['patient_name'].to_numpy()  

# ============================================================
# 🔹 NEW BLOCK — ADD THIS RIGHT HERE (after QC, before split)
# ============================================================

# GAIT joint indices (must match your gait14_graph node order)
GAIT_JOINTS = [
    2, 5,     # eyes
    11, 12,   # shoulders
    23, 24,   # hips
    25, 26,   # knees
    27, 28,   # ankles
    29, 30,   # heels
    31, 32    # foot index
]

# Select only gait joints
#X_clean = X_clean[:, :, GAIT_JOINTS, :]   # (N, T, 14, 3)

print("After joint selection:", X_clean.shape)
# MUST be: (N_windows, T, 14, 3)

# ============================================================
# 🔹 END NEW BLOCK
# ============================================================

# -------------------------------
# 1. Patient-level train/test split
# -------------------------------
unique_patients = np.unique(patient_names_clean)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

# Mask windows by patient
train_mask = np.isin(patient_names_clean, train_patients)
test_mask  = np.isin(patient_names_clean, test_patients)

# Apply masks
X_train = X_clean[train_mask]
X_test  = X_clean[test_mask]
y_bin_train = y_binary_clean[train_mask]
y_bin_test  = y_binary_clean[test_mask]
y_ml_train  = y_multilabel_clean[train_mask]
y_ml_test   = y_multilabel_clean[test_mask]

print(f"Train windows: {X_train.shape[0]}, Test windows: {X_test.shape[0]}")

# -------------------------------
# 2. Transpose to ST-GCN format: (N, C=3, T, V)
# -------------------------------
X_train_stgcn = np.transpose(X_train, (0, 3, 1, 2))
X_test_stgcn  = np.transpose(X_test,  (0, 3, 1, 2))

# Add person dimension (M=1)
X_train_stgcn = X_train_stgcn[..., np.newaxis]
X_test_stgcn  = X_test_stgcn[..., np.newaxis]


print(f"ST-GCN input shape train: {X_train_stgcn.shape}, test: {X_test_stgcn.shape}")

# -------------------------------
# 3. Separate abnormal windows for multi-label model
# -------------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

print(f"Multi-label train windows: {X_ml_train_stgcn.shape[0]}, test windows: {X_ml_test_stgcn.shape[0]}")

After joint selection: (15001, 60, 14, 3)
Train windows: 11874, Test windows: 3127
ST-GCN input shape train: (11874, 3, 60, 14, 1), test: (3127, 3, 60, 14, 1)
Multi-label train windows: 460, test windows: 355


In [ ]:
neighbor_links = [
    # Head
    (0, 1),        # left eye ↔ right eye
    (0, 2),        # left eye ↔ left shoulder
    (1, 3),        # right eye ↔ right shoulder

    # Upper body
    (2, 3),        # shoulders
    (2, 4),        # left shoulder → left hip
    (3, 5),        # right shoulder → right hip
    (4, 5),        # hips

    # Left leg
    (4, 6),        # left hip → knee
    (6, 8),        # knee → ankle
    (8, 10),       # ankle → heel
    (10, 12),      # heel → foot index

    # Right leg
    (5, 7),        # right hip → knee
    (7, 9),        # knee → ankle
    (9, 11),       # ankle → heel
    (11, 13),      # heel → foot index
]


In [10]:
print("Current working dir:", os.getcwd())



Current working dir: /Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/notebooks


In [4]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# Add models folder to Python path
# ----------------------------
import os

# Print current directory
print("Current working dir:", os.getcwd())

# Add relative path to models folder
sys.path.append("../models")  # adjust the dots if you are in a subfolder

# Check contents
print(os.listdir("../models"))         # should show ['st_gcn']
print(os.listdir("../models/net"))  # should show ['__init__.py', 'st_gcn.py', 'gait14_graph.py']
# Now import

# Import ST-GCN and your custom graph
from net.st_gcn import Model
from net.gait14_graph import Graph as GaitGraph



# ----------------------------
# Dataset wrapper
# ----------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ----------------------------
# Prepare DataLoaders
# ----------------------------
batch_size = 32

train_loader = DataLoader(
    PoseDataset(X_train_stgcn, y_bin_train),
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    PoseDataset(X_test_stgcn, y_bin_test),
    batch_size=batch_size,
    shuffle=False
)

# ----------------------------
# Device
# ----------------------------
#device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Build ST-GCN model with your custom gait graph
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

binary_stgcn = Model(
    in_channels=3,
    num_class=1,
    graph_args={},                # nothing needed
    edge_importance_weighting=True
).to(device)


# ----------------------------
# Loss and optimizer
# ----------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(binary_stgcn.parameters(), lr=1e-3)

# ----------------------------
# Training loop
# ----------------------------
num_epochs = 10

for epoch in range(num_epochs):
    binary_stgcn.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = binary_stgcn(X_batch).squeeze(1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {total_loss/len(train_loader):.4f}")

# ----------------------------
# Evaluation
# ----------------------------
binary_stgcn.eval()
all_probs, all_targets = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = binary_stgcn(X_batch).squeeze(1)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())

all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)
preds = (all_probs > 0.5).astype(int)

# ----------------------------
# Quick metrics
# ----------------------------
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

acc = accuracy_score(all_targets, preds)
f1 = f1_score(all_targets, preds)
cm = confusion_matrix(all_targets, preds)

print(f"Test Accuracy: {acc:.4f}, F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)


Current working dir: /Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/notebooks
['pose_landmarker_lite.task', 'net', '.DS_Store', 'multi_label_abnormal_model.bin', 'graph.py', '.gitkeep', 'binary_model.bin', 'binary_model_full.bin', 'multi_model.bin', 'multi_label_model_full.bin']
['gait14_graph.py', '.DS_Store', '__init__.py', 'utils', '__pycache__', 'st_gcn.py']
[Epoch 1/10] Train Loss: 0.0527
[Epoch 2/10] Train Loss: 0.0247
[Epoch 3/10] Train Loss: 0.0254
[Epoch 4/10] Train Loss: 0.0228
[Epoch 5/10] Train Loss: 0.0117
[Epoch 6/10] Train Loss: 0.0090
[Epoch 7/10] Train Loss: 0.0071
[Epoch 8/10] Train Loss: 0.0144
[Epoch 9/10] Train Loss: 0.0213
[Epoch 10/10] Train Loss: 0.0075
Test Accuracy: 0.9920, F1: 0.9639
Confusion Matrix:
 [[2768    4]
 [  21  334]]


In [46]:
print(X_batch.shape)
# should be (N, 3, T, 14, 1)


torch.Size([32, 3, 60, 14, 1])


In [47]:
print(binary_stgcn.graph.num_node)
# must be 14


18


In [48]:
print(binary_stgcn.data_bn.running_mean.shape)
# must be (42,)


torch.Size([54])


In [6]:
# with early stopping 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score
import numpy as np

# ----------------------------
# Dataset wrapper
# ----------------------------
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ----------------------------
# Split train into train + validation
# ----------------------------
val_ratio = 0.2
n_train = len(X_train_stgcn)
n_val = int(n_train * val_ratio)
n_train_actual = n_train - n_val

train_dataset = PoseDataset(X_train_stgcn, y_bin_train)
train_subset, val_subset = random_split(train_dataset, [n_train_actual, n_val])

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)
test_loader  = DataLoader(PoseDataset(X_test_stgcn, y_bin_test), batch_size=32, shuffle=False)

# ----------------------------
# Device
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Model
# ----------------------------
binary_stgcn = Model(
    in_channels=3,
    num_class=1,
    edge_importance_weighting=True,
    graph_args={}  # pass your gait graph class here
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(binary_stgcn.parameters(), lr=1e-3)

# ----------------------------
# Training loop with validation & early stopping
# ----------------------------
num_epochs = 20
best_f1 = 0
patience = 5
no_improve_counter = 0
best_model_path = "best_model.pt"

for epoch in range(num_epochs):
    # ----- Training -----
    binary_stgcn.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = binary_stgcn(X_batch).squeeze(1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # ----- Validation -----
    binary_stgcn.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            logits = binary_stgcn(X_val).squeeze(1)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()
            all_preds.append(preds.cpu())
            all_targets.append(y_val.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    val_f1 = f1_score(all_targets, all_preds)

    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f}, Val F1: {val_f1:.4f}")

    # ----- Save best model -----
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(binary_stgcn.state_dict(), best_model_path)
        no_improve_counter = 0
    else:
        no_improve_counter += 1

    # ----- Early stopping -----
    if no_improve_counter >= patience:
        print(f"No improvement for {patience} epochs. Stopping early.")
        break

# ----------------------------
# Load the best model
# ----------------------------
binary_stgcn.load_state_dict(torch.load(best_model_path))
binary_stgcn.eval()


[Epoch 1/20] Train Loss: 0.0717, Val F1: 0.9198
[Epoch 2/20] Train Loss: 0.0266, Val F1: 0.9368
[Epoch 3/20] Train Loss: 0.0252, Val F1: 0.9789
[Epoch 4/20] Train Loss: 0.0170, Val F1: 0.9630
[Epoch 5/20] Train Loss: 0.0132, Val F1: 0.9153
[Epoch 6/20] Train Loss: 0.0114, Val F1: 0.9412
[Epoch 7/20] Train Loss: 0.0116, Val F1: 0.9630
[Epoch 8/20] Train Loss: 0.0096, Val F1: 0.9677
No improvement for 5 epochs. Stopping early.


Model(
  (data_bn): BatchNorm1d(42, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (st_gcn_networks): ModuleList(
    (0): st_gcn(
      (gcn): ConvTemporalGraphical(
        (conv): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
      )
      (tcn): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0))
        (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (4): Dropout(p=0, inplace=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1-3): 3 x st_gcn(
      (gcn): ConvTemporalGraphical(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
      )
      (tcn): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(9, 1), s

In [10]:
# save the model and the stade:dict
save_path = "../models/binary_stgcn_gait14_stadedict.bin"
torch.save(binary_stgcn.state_dict(), save_path)
print(f"Model saved to {save_path}")

# save the model
#torch.save(binary_stgcn, "../models/binary_stgcn_model.bin")  # ❌ not recommended


Model saved to ../models/binary_stgcn_gait14_stadedict.bin


In [ ]:
# how to load the state_dict
binary_stgcn = Model(
    in_channels=3,
    num_class=1,
    edge_importance_weighting=True,
    graph_args={}   # same graph config as training
).to(device)

binary_stgcn.load_state_dict(
    torch.load("../models/binary_stgcn_model.bin", map_location=device)
)

binary_stgcn.eval()


In [ ]:
#minimal inference script
import torch
from net.st_gcn import Model
from net.gait14_graph import Graph as GaitGraph

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Model(
    in_channels=3,
    num_class=1,
    edge_importance_weighting=True,
    graph_args={}
).to(device)

model.load_state_dict(
    torch.load("binary_stgcn_weights.bin", map_location=device)
)

model.eval()


In [ ]:
# evaluatio of binary model with window and patient level metrics
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

# ----------------------------
# Dataset wrapper with patient IDs
# ----------------------------
class PoseDatasetWithPatient(Dataset):
    def __init__(self, X, y, patient_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = np.array(patient_ids)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]

# ----------------------------
# DataLoader
# ----------------------------
batch_size = 32
test_dataset = PoseDatasetWithPatient(X_test_stgcn, y_bin_test, patient_names_clean[test_mask])
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ----------------------------
# Evaluation
# ----------------------------
binary_stgcn.eval()
all_probs, all_targets, all_patients = [], [], []

with torch.no_grad():
    for X_batch, y_batch, pids in test_loader:
        X_batch = X_batch.to(device)
        logits = binary_stgcn(X_batch).squeeze(1)
        probs = torch.sigmoid(logits)
        
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())
        
        if isinstance(pids, torch.Tensor):
            all_patients.append(pids.numpy())
        else:
            all_patients.append(np.array(pids))

# Flatten lists
all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)
all_patients = np.concatenate(all_patients)

# ----------------------------
# Window-level metrics
# ----------------------------
window_preds = (all_probs > 0.5).astype(int)
acc_window = accuracy_score(all_targets, window_preds)
f1_window = f1_score(all_targets, window_preds)
precision_window = precision_score(all_targets, window_preds)
recall_window = recall_score(all_targets, window_preds)
try:
    auc_window = roc_auc_score(all_targets, all_probs)
except:
    auc_window = np.nan
cm_window = confusion_matrix(all_targets, window_preds)

print("=== Window-level metrics ===")
print(f"Accuracy:  {acc_window:.4f}")
print(f"F1-score:  {f1_window:.4f}")
print(f"Precision: {precision_window:.4f}")
print(f"Recall:    {recall_window:.4f}")
print(f"AUROC:     {auc_window:.4f}")
print("Confusion Matrix:\n", cm_window)

# ----------------------------
# Patient-level metrics
# ----------------------------
# Aggregate predictions per patient (majority vote)
patient_metrics = {}
unique_patients = np.unique(all_patients)
for pid in unique_patients:
    mask = all_patients == pid
    pred_majority = int(np.round(np.mean(window_preds[mask])))
    true_label = int(np.round(np.mean(all_targets[mask])))
    patient_metrics[pid] = (pred_majority, true_label)

# Convert to arrays
patient_preds = np.array([v[0] for v in patient_metrics.values()])
patient_true  = np.array([v[1] for v in patient_metrics.values()])

acc_patient = accuracy_score(patient_true, patient_preds)
f1_patient  = f1_score(patient_true, patient_preds)
precision_patient = precision_score(patient_true, patient_preds)
recall_patient = recall_score(patient_true, patient_preds)
try:
    auc_patient = roc_auc_score(patient_true, patient_preds)
except:
    auc_patient = np.nan
cm_patient = confusion_matrix(patient_true, patient_preds)

print("\n=== Patient-level metrics ===")
print(f"Accuracy:  {acc_patient:.4f}")
print(f"F1-score:  {f1_patient:.4f}")
print(f"Precision: {precision_patient:.4f}")
print(f"Recall:    {recall_patient:.4f}")
print(f"AUROC:     {auc_patient:.4f}")
print("Confusion Matrix:\n", cm_patient)


=== Window-level metrics ===
Accuracy:  0.9821
F1-score:  0.9159
Precision: 0.9807
Recall:    0.8592
AUROC:     0.9962
Confusion Matrix:
 [[2766    6]
 [  50  305]]

=== Patient-level metrics ===
Accuracy:  1.0000
F1-score:  1.0000
Precision: 1.0000
Recall:    1.0000
AUROC:     1.0000
Confusion Matrix:
 [[77  0]
 [ 0  5]]


## Binary model addressing the imbalance of the dataset better by weighting abnormal prediction more (increasing a higher loss - bad)

In [16]:
# ==============================
# ST-GCN Retraining + Evaluation Pipeline
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)
import numpy as np

# ----------------------------
# Dataset wrapper with patient IDs
# ----------------------------
class PoseDatasetWithPatient(Dataset):
    def __init__(self, X, y, patient_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = np.array(patient_ids)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]

# ----------------------------
# Create train/validation/test splits
# ----------------------------
val_ratio = 0.2
train_dataset = PoseDatasetWithPatient(X_train_stgcn, y_bin_train, patient_names_clean[train_mask])
n_val = int(len(train_dataset) * val_ratio)
n_train_actual = len(train_dataset) - n_val
train_subset, val_subset = random_split(train_dataset, [n_train_actual, n_val])

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)
test_loader  = DataLoader(PoseDatasetWithPatient(X_test_stgcn, y_bin_test, patient_names_clean[test_mask]),
                          batch_size=32, shuffle=False)

# ----------------------------
# Device
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Model
# ----------------------------
binary_stgcn = Model(
    in_channels=3,
    num_class=1,
    edge_importance_weighting=True,
    graph_args={}  # pass your gait graph class
).to(device)

# ----------------------------
# Weighted BCE loss (penalize missing abnormal windows more)
# ----------------------------
pos_weight = torch.tensor([(y_bin_train == 0).sum() / (y_bin_train == 1).sum()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(binary_stgcn.parameters(), lr=1e-3)

# ----------------------------
# Training loop with validation & early stopping
# ----------------------------
num_epochs = 50
best_f1 = 0
patience = 7
no_improve_counter = 0
best_model_path = "best_binary_stgcn.pt"

for epoch in range(num_epochs):
    # ----- Training -----
    binary_stgcn.train()
    total_loss = 0
    for X_batch, y_batch, _ in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = binary_stgcn(X_batch).squeeze(1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # ----- Validation -----
    binary_stgcn.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_val, y_val, _ in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            logits = binary_stgcn(X_val).squeeze(1)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()
            all_preds.append(preds.cpu())
            all_targets.append(y_val.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    val_f1 = f1_score(all_targets, all_preds)

    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f}, Val F1: {val_f1:.4f}")

    # ----- Save best model -----
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(binary_stgcn.state_dict(), best_model_path)
        no_improve_counter = 0
    else:
        no_improve_counter += 1

    # ----- Early stopping -----
    if no_improve_counter >= patience:
        print(f"No improvement for {patience} epochs. Stopping early.")
        break

# ----------------------------
# Load best model
# ----------------------------
binary_stgcn.load_state_dict(torch.load(best_model_path))
binary_stgcn.eval()

# ==============================
# Evaluation (window & patient level)
# ==============================

all_probs, all_targets, all_patients = [], [], []

with torch.no_grad():
    for X_batch, y_batch, pids in test_loader:
        X_batch = X_batch.to(device)
        logits = binary_stgcn(X_batch).squeeze(1)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())
        all_patients.append(np.array(pids))

# Flatten lists
all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)
all_patients = np.concatenate(all_patients)

# ----- Window-level -----
window_preds = (all_probs > 0.5).astype(int)
acc_window = accuracy_score(all_targets, window_preds)
f1_window = f1_score(all_targets, window_preds)
precision_window = precision_score(all_targets, window_preds)
recall_window = recall_score(all_targets, window_preds)
try:
    auc_window = roc_auc_score(all_targets, all_probs)
except:
    auc_window = np.nan
cm_window = confusion_matrix(all_targets, window_preds)

print("=== Window-level metrics ===")
print(f"Accuracy:  {acc_window:.4f}")
print(f"F1-score:  {f1_window:.4f}")
print(f"Precision: {precision_window:.4f}")
print(f"Recall:    {recall_window:.4f}")
print(f"AUROC:     {auc_window:.4f}")
print("Confusion Matrix:\n", cm_window)

# ----- Patient-level -----
unique_patients = np.unique(all_patients)
patient_preds = []
patient_probs = []
patient_true = []

for pid in unique_patients:
    mask = all_patients == pid
    # majority vote for prediction
    pred_majority = int(np.round(np.mean(window_preds[mask])))
    patient_preds.append(pred_majority)
    # max probability for AUROC
    patient_probs.append(all_probs[mask].max())
    patient_true.append(int(np.round(np.mean(all_targets[mask]))))

patient_preds = np.array(patient_preds)
patient_probs = np.array(patient_probs)
patient_true  = np.array(patient_true)

acc_patient = accuracy_score(patient_true, patient_preds)
f1_patient  = f1_score(patient_true, patient_preds)
precision_patient = precision_score(patient_true, patient_preds)
recall_patient = recall_score(patient_true, patient_preds)
try:
    auc_patient = roc_auc_score(patient_true, patient_probs)
except:
    auc_patient = np.nan
cm_patient = confusion_matrix(patient_true, patient_preds)

print("\n=== Patient-level metrics ===")
print(f"Accuracy:  {acc_patient:.4f}")
print(f"F1-score:  {f1_patient:.4f}")
print(f"Precision: {precision_patient:.4f}")
print(f"Recall:    {recall_patient:.4f}")
print(f"AUROC:     {auc_patient:.4f}")
print("Confusion Matrix:\n", cm_patient)


[Epoch 1/50] Train Loss: 0.5262, Val F1: 0.8247
[Epoch 2/50] Train Loss: 0.2192, Val F1: 0.8293
[Epoch 3/50] Train Loss: 0.1543, Val F1: 0.9551
[Epoch 4/50] Train Loss: 0.2058, Val F1: 0.8081
[Epoch 5/50] Train Loss: 0.1037, Val F1: 0.7257
[Epoch 6/50] Train Loss: 0.0649, Val F1: 0.6905
[Epoch 7/50] Train Loss: 0.1070, Val F1: 0.8657
[Epoch 8/50] Train Loss: 0.0784, Val F1: 0.8832
[Epoch 9/50] Train Loss: 0.0876, Val F1: 0.9457
[Epoch 10/50] Train Loss: 0.0631, Val F1: 0.9005
No improvement for 7 epochs. Stopping early.
=== Window-level metrics ===
Accuracy:  0.9878
F1-score:  0.9451
Precision: 0.9703
Recall:    0.9211
AUROC:     0.9982
Confusion Matrix:
 [[2762   10]
 [  28  327]]

=== Patient-level metrics ===
Accuracy:  1.0000
F1-score:  1.0000
Precision: 1.0000
Recall:    1.0000
AUROC:     1.0000
Confusion Matrix:
 [[77  0]
 [ 0  5]]


In [17]:
# save the model and the stade:dict
save_path = "../models/binary_stgcn_gait14_improved_stadedict.bin"
torch.save(binary_stgcn.state_dict(), save_path)
print(f"Model saved to {save_path}")

Model saved to ../models/binary_stgcn_gait14_improved_stadedict.bin


# Multilabel classification with the ST-GCN model from the website

In [18]:
# %%
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)
import numpy as np

# ----------------------------
# Dataset wrapper for multi-label model
# ----------------------------
class PoseDatasetWithPatient(Dataset):
    def __init__(self, X, y, patient_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = np.array(patient_ids)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]

# ----------------------------
# Select only abnormal windows
# ----------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_ml_train[abnormal_mask_train]

X_ml_test_stgcn  = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_ml_test[abnormal_mask_test]

patient_train_filtered = patient_names_clean[train_mask][abnormal_mask_train]
patient_test_filtered  = patient_names_clean[test_mask][abnormal_mask_test]

# ----------------------------
# Split train into train + validation
# ----------------------------
val_ratio = 0.2
n_train = len(X_ml_train_stgcn)
n_val = int(n_train * val_ratio)
n_train_actual = n_train - n_val

ml_dataset = PoseDatasetWithPatient(X_ml_train_stgcn, y_ml_train_filtered, patient_train_filtered)
train_subset, val_subset = random_split(ml_dataset, [n_train_actual, n_val])

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)
test_dataset = PoseDatasetWithPatient(X_ml_test_stgcn, y_ml_test_filtered, patient_test_filtered)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ----------------------------
# Device
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Model
# ----------------------------
num_classes = y_ml_train_filtered.shape[1]

# Compute pos_weight for each label
pos_weight = torch.tensor([
    (len(y_ml_train_filtered) - y_ml_train_filtered[:, i].sum()) / y_ml_train_filtered[:, i].sum()
    for i in range(num_classes)
], dtype=torch.float32).to(device)

multilabel_stgcn = Model(
    in_channels=3,
    num_class=num_classes,
    edge_importance_weighting=True,
    graph_args={}  # pass your gait graph class here
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(multilabel_stgcn.parameters(), lr=1e-3)

# ----------------------------
# Training loop with early stopping
# ----------------------------
num_epochs = 50
best_f1 = 0
patience = 7
no_improve_counter = 0
best_model_path = "best_multilabel_model.pt"

for epoch in range(num_epochs):
    # ----- Training -----
    multilabel_stgcn.train()
    total_loss = 0
    for X_batch, y_batch, _ in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = multilabel_stgcn(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # ----- Validation -----
    multilabel_stgcn.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_val, y_val, _ in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            logits = multilabel_stgcn(X_val)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()
            all_preds.append(preds.cpu())
            all_targets.append(y_val.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    val_f1 = f1_score(all_targets, all_preds, average="macro")

    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f}, Val F1: {val_f1:.4f}")

    # ----- Save best model -----
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(multilabel_stgcn.state_dict(), best_model_path)
        no_improve_counter = 0
    else:
        no_improve_counter += 1

    # ----- Early stopping -----
    if no_improve_counter >= patience:
        print(f"No improvement for {patience} epochs. Stopping early.")
        break

# ----------------------------
# Load best model
# ----------------------------
multilabel_stgcn.load_state_dict(torch.load(best_model_path))
multilabel_stgcn.eval()

# ----------------------------
# Evaluation (window + patient level with soft voting)
# ----------------------------
all_probs, all_targets, all_patients = [], [], []

with torch.no_grad():
    for X_batch, y_batch, pids in test_loader:
        X_batch = X_batch.to(device)
        logits = multilabel_stgcn(X_batch)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())
        all_patients.append(np.array(pids))

all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)
all_patients = np.concatenate(all_patients)

# ----------------------------
# Window-level metrics
# ----------------------------
window_preds = (all_probs > 0.5).astype(int)
acc_window = accuracy_score(all_targets, window_preds)
f1_window = f1_score(all_targets, window_preds, average="macro")
precision_window = precision_score(all_targets, window_preds, average="macro")
recall_window = recall_score(all_targets, window_preds, average="macro")
try:
    auc_window = roc_auc_score(all_targets, all_probs, average="macro")
except:
    auc_window = np.nan
cm_window = confusion_matrix(all_targets.argmax(axis=1), window_preds.argmax(axis=1))

print("=== Window-level metrics ===")
print(f"Accuracy:  {acc_window:.4f}")
print(f"F1-score:  {f1_window:.4f}")
print(f"Precision: {precision_window:.4f}")
print(f"Recall:    {recall_window:.4f}")
print(f"AUROC:     {auc_window:.4f}")
print("Confusion Matrix:\n", cm_window)

# ----------------------------
# Patient-level metrics (soft voting)
# ----------------------------
unique_patients = np.unique(all_patients)
patient_preds, patient_true = [], []

for pid in unique_patients:
    mask = all_patients == pid
    # Soft voting: average probabilities per patient, then threshold
    avg_probs = np.mean(all_probs[mask], axis=0)
    pred_label = (avg_probs > 0.5).astype(int)
    true_label = all_targets[mask][0]  # all windows have same true label
    patient_preds.append(pred_label)
    patient_true.append(true_label)

patient_preds = np.array(patient_preds)
patient_true  = np.array(patient_true)

acc_patient = accuracy_score(patient_true, patient_preds)
f1_patient  = f1_score(patient_true, patient_preds, average="macro")
precision_patient = precision_score(patient_true, patient_preds, average="macro")
recall_patient = recall_score(patient_true, patient_preds, average="macro")
try:
    auc_patient = roc_auc_score(patient_true, patient_preds, average="macro")
except:
    auc_patient = np.nan
cm_patient = confusion_matrix(patient_true.argmax(axis=1), patient_preds.argmax(axis=1))

print("\n=== Patient-level metrics (soft voting) ===")
print(f"Accuracy:  {acc_patient:.4f}")
print(f"F1-score:  {f1_patient:.4f}")
print(f"Precision: {precision_patient:.4f}")
print(f"Recall:    {recall_patient:.4f}")
print(f"AUROC:     {auc_patient:.4f}")
print("Confusion Matrix:\n", cm_patient)


[Epoch 1/50] Train Loss: 0.9981, Val F1: 0.3188
[Epoch 2/50] Train Loss: 0.7337, Val F1: 0.3509
[Epoch 3/50] Train Loss: 0.5714, Val F1: 0.3854
[Epoch 4/50] Train Loss: 0.5962, Val F1: 0.5620
[Epoch 5/50] Train Loss: 0.5879, Val F1: 0.5487
[Epoch 6/50] Train Loss: 0.4731, Val F1: 0.5990
[Epoch 7/50] Train Loss: 0.4365, Val F1: 0.6911
[Epoch 8/50] Train Loss: 0.3722, Val F1: 0.7819
[Epoch 9/50] Train Loss: 0.2988, Val F1: 0.7231
[Epoch 10/50] Train Loss: 0.3005, Val F1: 0.7190
[Epoch 11/50] Train Loss: 0.2997, Val F1: 0.7762
[Epoch 12/50] Train Loss: 0.2720, Val F1: 0.7303
[Epoch 13/50] Train Loss: 0.2393, Val F1: 0.7268
[Epoch 14/50] Train Loss: 0.2762, Val F1: 0.8018
[Epoch 15/50] Train Loss: 0.2619, Val F1: 0.7933
[Epoch 16/50] Train Loss: 0.2268, Val F1: 0.6797
[Epoch 17/50] Train Loss: 0.2304, Val F1: 0.8793
[Epoch 18/50] Train Loss: 0.2386, Val F1: 0.8027
[Epoch 19/50] Train Loss: 0.2731, Val F1: 0.8030
[Epoch 20/50] Train Loss: 0.2436, Val F1: 0.7849
[Epoch 21/50] Train Loss: 0.2

/Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. 

In [19]:
# Count positives per label
print("Train label counts:\n", y_ml_train_filtered.sum(axis=0))
print("Test label counts:\n", y_ml_test_filtered.sum(axis=0))


Train label counts:
 [110  51 155  49  75]
Test label counts:
 [ 47 130   0 151  83]


In [22]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)

# ----------------------------
# Dataset wrapper with patient IDs
# ----------------------------
class PoseDatasetWithPatient(Dataset):
    def __init__(self, X, y, patient_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = np.array(patient_ids)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]

# ----------------------------
# Compute pos_weight for multilabel
# ----------------------------
train_label_counts = np.sum(y_ml_train_filtered, axis=0)
train_label_counts = np.where(train_label_counts == 0, 1, train_label_counts)  # avoid div by 0
pos_weight = torch.tensor((len(y_ml_train_filtered) - train_label_counts) / train_label_counts, dtype=torch.float32).to("cuda" if torch.cuda.is_available() else "cpu")
print("Pos weight per label:", pos_weight)

# ----------------------------
# Split train into train + validation
# ----------------------------
val_ratio = 0.2
n_train = len(X_ml_train_stgcn)
n_val = int(n_train * val_ratio)
n_train_actual = n_train - n_val

train_dataset = PoseDatasetWithPatient(X_ml_train_stgcn, y_ml_train_filtered, patient_names_clean[train_mask][abnormal_mask_train])
train_subset, val_subset = random_split(train_dataset, [n_train_actual, n_val])

batch_size = 32
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_dataset = PoseDatasetWithPatient(X_ml_test_stgcn, y_ml_test_filtered, patient_names_clean[test_mask][abnormal_mask_test])
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ----------------------------
# Device
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Model
# ----------------------------
binary_stgcn = Model(
    in_channels=3,
    num_class=y_ml_train_filtered.shape[1],  # 5 labels
    edge_importance_weighting=True,
    graph_args={}  # your gait graph
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(binary_stgcn.parameters(), lr=1e-3)

# ----------------------------
# Training loop with validation & early stopping
# ----------------------------
num_epochs = 50
best_f1 = 0
patience = 7
no_improve_counter = 0
best_model_path = "best_multilabel_model.pt"

for epoch in range(num_epochs):
    # ----- Training -----
    binary_stgcn.train()
    total_loss = 0
    for X_batch, y_batch, _ in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = binary_stgcn(X_batch).squeeze()
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # ----- Validation -----
    binary_stgcn.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_val, y_val, _ in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            logits = binary_stgcn(X_val).squeeze()
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            all_preds.append(preds.cpu())
            all_targets.append(y_val.cpu())
    
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    val_f1 = f1_score(all_targets, all_preds, average='macro')  # macro works for multilabel

    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f}, Val F1: {val_f1:.4f}")

    # ----- Save best model -----
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(binary_stgcn.state_dict(), best_model_path)
        no_improve_counter = 0
    else:
        no_improve_counter += 1

    # ----- Early stopping -----
    if no_improve_counter >= patience:
        print(f"No improvement for {patience} epochs. Stopping early.")
        break

# ----------------------------
# Load best model
# ----------------------------
binary_stgcn.load_state_dict(torch.load(best_model_path))
binary_stgcn.eval()

# ----------------------------
# Evaluation
# ----------------------------
all_probs, all_targets, all_patients = [], [], []
with torch.no_grad():
    for X_batch, y_batch, pids in test_loader:
        X_batch = X_batch.to(device)
        logits = binary_stgcn(X_batch).squeeze()
        probs = torch.sigmoid(logits)
        
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())
        all_patients.append(np.array(pids))

all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)
all_patients = np.concatenate(all_patients)

# ----------------------------
# Window-level metrics
# ----------------------------
window_preds = (all_probs > 0.5).astype(int)
acc_window = accuracy_score(all_targets, window_preds)
f1_window = f1_score(all_targets, window_preds, average='macro')
precision_window = precision_score(all_targets, window_preds, average='macro', zero_division=0)
recall_window = recall_score(all_targets, window_preds, average='macro', zero_division=0)
try:
    auc_window = roc_auc_score(all_targets, all_probs, average='macro')
except:
    auc_window = np.nan
cm_window = confusion_matrix(all_targets.argmax(axis=1), window_preds.argmax(axis=1))

print("\n=== Window-level metrics ===")
print(f"Accuracy:  {acc_window:.4f}")
print(f"F1-score:  {f1_window:.4f}")
print(f"Precision: {precision_window:.4f}")
print(f"Recall:    {recall_window:.4f}")
print(f"AUROC:     {auc_window:.4f}")
print("Confusion Matrix:\n", cm_window)

# ----------------------------
# Patient-level metrics (soft voting)
# ----------------------------
unique_patients = np.unique(all_patients)
patient_preds, patient_true = [], []

for pid in unique_patients:
    mask = all_patients == pid
    avg_probs = np.mean(all_probs[mask], axis=0)  # soft voting
    pred = (avg_probs > 0.5).astype(int)
    true = np.round(np.mean(all_targets[mask], axis=0)).astype(int)
    patient_preds.append(pred)
    patient_true.append(true)

patient_preds = np.array(patient_preds)
patient_true = np.array(patient_true)

acc_patient = accuracy_score(patient_true, patient_preds)
f1_patient  = f1_score(patient_true, patient_preds, average='macro')
precision_patient = precision_score(patient_true, patient_preds, average='macro', zero_division=0)
recall_patient = recall_score(patient_true, patient_preds, average='macro', zero_division=0)
try:
    auc_patient = roc_auc_score(patient_true, patient_preds, average='macro')
except:
    auc_patient = np.nan
cm_patient = confusion_matrix(patient_true.argmax(axis=1), patient_preds.argmax(axis=1))

print("\n=== Patient-level metrics (soft voting) ===")
print(f"Accuracy:  {acc_patient:.4f}")
print(f"F1-score:  {f1_patient:.4f}")
print(f"Precision: {precision_patient:.4f}")
print(f"Recall:    {recall_patient:.4f}")
print(f"AUROC:     {auc_patient:.4f}")
print("Confusion Matrix:\n", cm_patient)


Pos weight per label: tensor([3.1818, 8.0196, 1.9677, 8.3878, 5.1333])
[Epoch 1/50] Train Loss: 0.9684, Val F1: 0.2614
[Epoch 2/50] Train Loss: 0.6424, Val F1: 0.3059
[Epoch 3/50] Train Loss: 0.5540, Val F1: 0.4945
[Epoch 4/50] Train Loss: 0.4866, Val F1: 0.5241
[Epoch 5/50] Train Loss: 0.4411, Val F1: 0.5742
[Epoch 6/50] Train Loss: 0.3775, Val F1: 0.5731
[Epoch 7/50] Train Loss: 0.3247, Val F1: 0.6238
[Epoch 8/50] Train Loss: 0.2829, Val F1: 0.6789
[Epoch 9/50] Train Loss: 0.2769, Val F1: 0.7057
[Epoch 10/50] Train Loss: 0.3022, Val F1: 0.6215
[Epoch 11/50] Train Loss: 0.2929, Val F1: 0.7170
[Epoch 12/50] Train Loss: 0.3518, Val F1: 0.6445
[Epoch 13/50] Train Loss: 0.3533, Val F1: 0.6661
[Epoch 14/50] Train Loss: 0.2691, Val F1: 0.6229
[Epoch 15/50] Train Loss: 0.1932, Val F1: 0.7288
[Epoch 16/50] Train Loss: 0.1621, Val F1: 0.7074
[Epoch 17/50] Train Loss: 0.1752, Val F1: 0.7535
[Epoch 18/50] Train Loss: 0.1893, Val F1: 0.8071
[Epoch 19/50] Train Loss: 0.2108, Val F1: 0.7185
[Epoch 

/Users/marcbp/spiced_bootcamp/Capstone Project/GAITy-Capstone-Modeling/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))


In [26]:
# %%
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)

# ----------------------------
# Dataset wrapper with patient IDs
# ----------------------------
class PoseDatasetWithPatient(Dataset):
    def __init__(self, X, y, patient_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.patient_ids = np.array(patient_ids)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.patient_ids[idx]

# ----------------------------
# Filter abnormal windows only
# ----------------------------
abnormal_mask_train = y_bin_train == 1
abnormal_mask_test  = y_bin_test == 1

X_ml_train_stgcn = X_train_stgcn[abnormal_mask_train]
y_ml_train_filtered = y_multilabel_clean[train_mask][abnormal_mask_train]
train_patient_ids = np.array(patient_names_clean[train_mask])[abnormal_mask_train]

X_ml_test_stgcn = X_test_stgcn[abnormal_mask_test]
y_ml_test_filtered = y_multilabel_clean[test_mask][abnormal_mask_test]
test_patient_ids = np.array(patient_names_clean[test_mask])[abnormal_mask_test]

print("Train abnormal windows:", X_ml_train_stgcn.shape[0])
print("Test abnormal windows:", X_ml_test_stgcn.shape[0])

# ----------------------------
# Compute per-label pos_weight
# ----------------------------
pos_weight = torch.tensor(
    [(y_ml_train_filtered[:, i] == 0).sum() / (y_ml_train_filtered[:, i] == 1).sum()
     for i in range(y_ml_train_filtered.shape[1])],
    dtype=torch.float32
)
print("Pos weight per label:", pos_weight)

# ----------------------------
# Train/validation split
# ----------------------------
val_ratio = 0.2
n_train = len(X_ml_train_stgcn)
n_val = int(n_train * val_ratio)
n_train_actual = n_train - n_val

train_dataset = PoseDatasetWithPatient(X_ml_train_stgcn, y_ml_train_filtered, train_patient_ids)
train_subset, val_subset = random_split(train_dataset, [n_train_actual, n_val])

train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_subset, batch_size=16, shuffle=False)
test_loader  = DataLoader(PoseDatasetWithPatient(X_ml_test_stgcn, y_ml_test_filtered, test_patient_ids),
                          batch_size=16, shuffle=False)

# ----------------------------
# Device
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Model
# ----------------------------
binary_stgcn = Model(
    in_channels=3,
    num_class=y_ml_train_filtered.shape[1],  # multilabel
    edge_importance_weighting=True,
    graph_args={}  # pass your GaitGraph class here
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = optim.Adam(binary_stgcn.parameters(), lr=1e-3)

# ----------------------------
# Training loop with validation & early stopping
# ----------------------------
num_epochs = 50
best_f1 = 0
patience = 7
no_improve_counter = 0
best_model_path = "best_multilabel_model.pt"

for epoch in range(num_epochs):
    # ----- Training -----
    binary_stgcn.train()
    total_loss = 0
    for X_batch, y_batch, _ in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = binary_stgcn(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # ----- Validation -----
    binary_stgcn.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_val, y_val, _ in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            logits = binary_stgcn(X_val)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()
            all_preds.append(preds.cpu())
            all_targets.append(y_val.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    val_f1 = f1_score(all_targets, all_preds, average='macro')

    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f}, Val F1: {val_f1:.4f}")

    # ----- Save best model -----
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(binary_stgcn.state_dict(), best_model_path)
        no_improve_counter = 0
    else:
        no_improve_counter += 1

    # ----- Early stopping -----
    if no_improve_counter >= patience:
        print(f"No improvement for {patience} epochs. Stopping early.")
        break

# ----------------------------
# Load the best model
# ----------------------------
binary_stgcn.load_state_dict(torch.load(best_model_path))
binary_stgcn.eval()

# ----------------------------
# Evaluation: Window-level + Patient-level (soft voting)
# ----------------------------
all_probs, all_targets, all_patients = [], [], []

with torch.no_grad():
    for X_batch, y_batch, pids in test_loader:
        X_batch = X_batch.to(device)
        logits = binary_stgcn(X_batch)
        probs = torch.sigmoid(logits)
        
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.numpy())
        if isinstance(pids, torch.Tensor):
            all_patients.append(pids.numpy())
        else:
            all_patients.append(np.array(pids))

all_probs = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets)
all_patients = np.concatenate(all_patients)

# ----------------------------
# Window-level metrics
# ----------------------------
window_preds = (all_probs > 0.5).astype(int)
acc_window = accuracy_score(all_targets, window_preds)
f1_window = f1_score(all_targets, window_preds, average='macro')
precision_window = precision_score(all_targets, window_preds, average='macro', zero_division=0)
recall_window = recall_score(all_targets, window_preds, average='macro', zero_division=0)
try:
    auc_window = roc_auc_score(all_targets, all_probs, average='macro')
except:
    auc_window = np.nan
cm_window = confusion_matrix(all_targets.argmax(axis=1), window_preds.argmax(axis=1))

print("\n=== Window-level metrics ===")
print(f"Accuracy:  {acc_window:.4f}")
print(f"F1-score:  {f1_window:.4f}")
print(f"Precision: {precision_window:.4f}")
print(f"Recall:    {recall_window:.4f}")
print(f"AUROC:     {auc_window:.4f}")
print("Confusion Matrix:\n", cm_window)

# ----------------------------
# Patient-level metrics (soft voting)
# ----------------------------
unique_patients = np.unique(all_patients)
patient_preds, patient_true = [], []

for pid in unique_patients:
    mask = all_patients == pid
    # Soft voting: average predicted probabilities across windows
    pred_avg = all_probs[mask].mean(axis=0)
    true_label = all_targets[mask].max(axis=0)  # if any window has label=1, patient label=1
    patient_preds.append((pred_avg > 0.5).astype(int))
    patient_true.append(true_label.astype(int))

patient_preds = np.array(patient_preds)
patient_true  = np.array(patient_true)

acc_patient = accuracy_score(patient_true, patient_preds)
f1_patient  = f1_score(patient_true, patient_preds, average='macro')
precision_patient = precision_score(patient_true, patient_preds, average='macro', zero_division=0)
recall_patient = recall_score(patient_true, patient_preds, average='macro', zero_division=0)
try:
    auc_patient = roc_auc_score(patient_true, patient_preds, average='macro')
except:
    auc_patient = np.nan
cm_patient = confusion_matrix(patient_true.argmax(axis=1), patient_preds.argmax(axis=1))

print("\n=== Patient-level metrics (soft voting) ===")
print(f"Accuracy:  {acc_patient:.4f}")
print(f"F1-score:  {f1_patient:.4f}")
print(f"Precision: {precision_patient:.4f}")
print(f"Recall:    {recall_patient:.4f}")
print(f"AUROC:     {auc_patient:.4f}")
print("Confusion Matrix:\n", cm_patient)


Train abnormal windows: 460
Test abnormal windows: 355
Pos weight per label: tensor([3.1818, 8.0196, 1.9677, 8.3878, 5.1333])
[Epoch 1/50] Train Loss: 0.9292, Val F1: 0.2517
[Epoch 2/50] Train Loss: 0.7888, Val F1: 0.5178
[Epoch 3/50] Train Loss: 0.6168, Val F1: 0.6592
[Epoch 4/50] Train Loss: 0.5577, Val F1: 0.5977
[Epoch 5/50] Train Loss: 0.5324, Val F1: 0.7649
[Epoch 6/50] Train Loss: 0.5313, Val F1: 0.6743
[Epoch 7/50] Train Loss: 0.5414, Val F1: 0.6529
[Epoch 8/50] Train Loss: 0.4943, Val F1: 0.6777
[Epoch 9/50] Train Loss: 0.4840, Val F1: 0.6881
[Epoch 10/50] Train Loss: 0.4729, Val F1: 0.6764
[Epoch 11/50] Train Loss: 0.4234, Val F1: 0.7844
[Epoch 12/50] Train Loss: 0.3408, Val F1: 0.7466
[Epoch 13/50] Train Loss: 0.3961, Val F1: 0.8435
[Epoch 14/50] Train Loss: 0.4010, Val F1: 0.7135
[Epoch 15/50] Train Loss: 0.3516, Val F1: 0.8442
[Epoch 16/50] Train Loss: 0.3489, Val F1: 0.8531
[Epoch 17/50] Train Loss: 0.3195, Val F1: 0.7703
[Epoch 18/50] Train Loss: 0.3961, Val F1: 0.6526
[

In [29]:
# Define a path
save_path = "../models/multilabel_stgcn_state_dict.pt"

# Save only the state dict
torch.save(binary_stgcn.state_dict(), save_path)
print(f"Model state dict saved to {save_path}")


Model state dict saved to ../models/multilabel_stgcn_state_dict.pt


In [ ]:
# Reload the model later
# Recreate the model architecture first
multilabel_stgcn = Model(
    in_channels=3,
    num_class=5,  # number of labels for multilabel classification
    edge_importance_weighting=True,
    graph_args={}  # your gait graph arguments
).to(device)

# Load the saved weights
multilabel_stgcn.load_state_dict(torch.load(save_path, map_location=device))
multilabel_stgcn.eval()  # set to evaluation mode
